# Generate synthetic dataset

In [1]:
import pathlib

import fakeitmakeit as fm
import numpy as np
import pandas as pd

data_dir = pathlib.Path("data/large_synthetic")
data_dir.mkdir(parents=True, exist_ok=True)

## Participants and sessions

In [2]:
n = 300
n_supervisors = 100
n_moderators = 6

supervisors = pd.Series(list(set(fm.name() for _ in range(n_supervisors))))
moderators = supervisors.sample(n_moderators, replace=False).to_numpy()

participant_2 = supervisors.sample(n, replace=True).to_numpy()
participant_3 = supervisors.sample(n, replace=True).to_numpy()


def who_is_the_chair(row):
    if row.participant_2 in moderators:
        return row.participant_2
    if row.participant_3 in moderators:
        return row.participant_3
    return pd.Series(moderators).sample(1, replace=True).iloc[0]


for i, p in enumerate(participant_3):
    candidate = p
    while candidate == participant_2[i]:
        candidate = supervisors.sample(1, replace=True).to_numpy()[0]
    participant_3[i] = candidate

presentations = (
    fm.cohort(n)
    .assign(
        participant_1=pd.col("first_name") + " " + pd.col("last_name"),
        participant_2=participant_2,
        participant_3=participant_3,
        chair=lambda df: df.apply(who_is_the_chair, axis=1),
    )
    .drop_duplicates(subset=["participant_1"])
    .loc[:, ["participant_1", "participant_2", "participant_3", "chair"]]
    .rename_axis("id")
)

# Participants no longer enforces any of this (a presentation's people are just an
# unordered, deduplicated list) - these are just generator-side choices, to keep the
# synthetic dataset looking like a plausible real conference rather than a stress
# test. A chair coinciding with one of the other three (self-chairing) is deliberately
# allowed here and deduplicated below when unrolling into participants.csv, same as
# any other repeated person would be.
assert presentations.participant_1.is_unique
assert presentations.participant_2.eq(presentations.participant_3).sum() == 0
assert presentations.participant_1.eq(presentations.participant_2).sum() == 0
assert presentations.participant_1.eq(presentations.participant_3).sum() == 0

print(f"Number of presentations: {len(presentations)}")
print(f"Number of unique participant_2: {len(presentations.participant_2.unique())}")
print(f"Number of unique participant_3: {len(presentations.participant_3.unique())}")
print(f"Number of unique chairs: {len(presentations.chair.unique())}")

# Unroll each presentation's four role columns into participants.csv (long format:
# one row per (id, participant) pair), deduplicating per presentation - a chair who
# is also one of the other three roles must appear only once, since Participants now
# enforces no repeats within a single presentation. sessions.csv reuses the chair as
# the session (room/track) label, one row per presentation.
participant_rows = []
session_rows = []
for presentation_id, row in presentations.iterrows():
    people = [row.participant_1, row.participant_2, row.participant_3, row.chair]
    seen = set()
    for person in people:
        if person not in seen:
            seen.add(person)
            participant_rows.append({"id": presentation_id, "participant": person})
    session_rows.append({"id": presentation_id, "session": row.chair})

participants = pd.DataFrame(participant_rows)
sessions = pd.DataFrame(session_rows).set_index("id")["session"]

participants.to_csv(data_dir / "participants.csv", index=False)
sessions.to_csv(data_dir / "sessions.csv", index=True)

participants.sample(20)

Number of presentations: 288
Number of unique participant_2: 95
Number of unique participant_3: 94
Number of unique chairs: 6


,id,participant
738,elr5242,Alejandro Rasmussen
1057,wg89,Erin Bernard
232,bkh12,Stephen Espinoza
1084,xc47,Alicia Lynch
688,gf38,Marc Wall
788,cmb8552,Charles Barr
478,xx7272,Kelly French
1102,lyw5590,Leonard Williams
745,dd266,Robin Bryant
491,lh129,Alexandra Rivera


## Unavailability table

In [3]:
# Availability exceptions table — one row per unavailability rule.
#
# Semantics of "day" and "slot" (both nullable, using pandas' Int64):
#   - day = <value>, slot = NaN      -> unavailable ALL DAY on that day
#   - day = NaN,      slot = <value> -> unavailable during that slot, EVERY day
#   - day = <value>,  slot = <value> -> unavailable for that specific slot only
#   - day = NaN,      slot = NaN     -> unavailable for the ENTIRE conference
#
# NaN acts as a wildcard meaning "applies to all values" for that column.
#
# Note: "global" (entire-conference) rules are deliberately not generated here. At
# this scale, a single person with a global restriction who is also required (as
# participant or chair) for any presentation makes that presentation permanently
# unschedulable, and with ~100 supervisors shared across ~292 presentations that's
# virtually guaranteed to happen - which would make this whole fixture infeasible.
# The dedicated small `unavailable_entire_conference` scenario under tests/data
# exercises that semantic deterministically instead.

n_restrictions = 60
rng = np.random.default_rng()

# Allow repeated people so one person can have multiple restrictions.
people = supervisors.sample(n_restrictions, replace=True).to_numpy()

# Mixed rule types (day-only, slot-only, specific-slot), each equally plausible.
rule_types = rng.choice(
    ["day_only", "slot_only", "specific_slot"],
    size=n_restrictions,
    p=[0.35, 0.35, 0.30],
)

days = np.full(n_restrictions, np.nan)
slots = np.full(n_restrictions, np.nan)

for i, rule in enumerate(rule_types):
    if rule == "day_only":
        days[i] = rng.integers(1, 8)
    elif rule == "slot_only":
        slots[i] = rng.integers(1, 9)
    elif rule == "specific_slot":
        days[i] = rng.integers(1, 8)
        slots[i] = rng.integers(1, 9)

unavailable = pd.DataFrame(
    {
        "person": people,
        "day": days,
        "slot": slots,
    }
).astype({"day": "Int64", "slot": "Int64"})

print(f"Number of restrictions: {len(unavailable)}")
repeat_counts = unavailable.person.value_counts()
print(f"People with multiple restrictions: {(repeat_counts > 1).sum()}")

unavailable.to_csv(data_dir / "unavailable.csv", index=False)

unavailable.sample(5)

Number of restrictions: 60
People with multiple restrictions: 13


,person,day,slot
4,Jennifer Gutierrez,<NA>,5
9,Kelly French,4,2
11,Alan Pearson,3,3
50,William Poole,4,<NA>
18,Amanda Wilson,<NA>,8
